# 📖 NovelReader Translate & Audio — Dịch truyện AI & TTS Audio (Google Colab T4 GPU)

Chạy hệ thống **NovelReader Translate & Audio** hoàn chỉnh ngay trên Google Colab — kết hợp mô hình dịch máy neural **HachimiMT / MoxhiMT (CTranslate2 INT8)** với bộ tổng hợp giọng đọc audio tiếng Việt **Piper TTS**, xuất cả file văn bản TXT/EPUB lẫn file audio MP3.

> ## ⚡ BẬT GPU T4 TRƯỚC (RẤT QUAN TRỌNG) — nhanh gấp 50–80 lần CPU!
> Notebook này đã được cấu hình sẵn để tự động yêu cầu GPU T4 từ Colab. Tuy nhiên nếu Colab chưa gán GPU, hãy bấm:
> **`Runtime → Change runtime type → T4 GPU`** → `Save`, rồi mới bấm **`Runtime → Run all`** (Ctrl+F9).
> *(Giao diện tiếng Việt: **`Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU → Lưu`**)*
>
> ⚠️ **Lưu ý:** Chọn đúng **`T4 GPU`**, KHÔNG chọn TPU (CTranslate2 chỉ hỗ trợ GPU NVIDIA/CUDA).
>
> 🚀 **Hiệu năng GPU T4:** Tốc độ dịch đạt **~40.000 ký tự/giây** (dịch toàn bộ cuốn truyện 2–3 triệu chữ chỉ mất khoảng **1–2 phút**).

---
**Cách sử dụng:**
1. Kiểm tra đã bật **T4 GPU**, sau đó bấm **`Thời gian chạy → Chạy tất cả`** (hoặc `Ctrl + F9`).
2. Ở **Bước 2**, trình tải sẽ tự động tải các gói `nvidia-cublas-cu12` (chứa `libcublas.so.12`) và bản **PyTorch CUDA** chuẩn xác nhất để kích hoạt GPU.
3. Ở **Bước 5**, bấm vào đường link công khai (`https://*.trycloudflare.com`) để mở Web UI trên trình duyệt máy tính hoặc điện thoại!

### 1. Tải mã nguồn NovelReader_Translate & Chuẩn bị thư mục

In [ ]:
# ==============================================================================
# BƯỚC 1: TẢI HOẶC CẬP NHẬT MÃ NGUỒN DỰ ÁN
# ==============================================================================
import os, sys, shutil

REPO_URL = "https://github.com/dungbeommm/NovelReader_Translate.git"
REPO_DIR = "/content/NovelReader_Translate"

if not os.path.exists(REPO_DIR):
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Thư mục {REPO_DIR} đã tồn tại, đang pull cập nhật mới nhất...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Tạo các thư mục cần thiết
for d in ["data/logs", "output_text", "output_audio", "models/translation", "models/voices"]:
    os.makedirs(d, exist_ok=True)

print("📁 Thư mục làm việc hiện tại:", os.getcwd())

### 2. Trình tải & Cài đặt PyTorch CUDA + Thư viện cuBLAS 12 (libcublas.so.12)

In [ ]:
# ==============================================================================
# BƯỚC 2: TRÌNH TẢI & CÀI ĐẶT PYTORCH CUDA + CUBLAS 12 + KÍCH HOẠT GPU T4
# ==============================================================================
import os, sys, subprocess

# 1. Cài đặt các gói thư viện runtime NVIDIA cuBLAS 12 (chứa libcublas.so.12 & libcublasLt.so.12)
print("📦 Đang kiểm tra & tải thư viện nvidia-cublas-cu12 nvidia-cudnn-cu12...")
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

# 2. Chạy trình kiểm tra và tải cài đặt PyTorch CUDA tự động
!python scripts/setup_gpu.py

# 3. Thiết lập các biến môi trường tăng tốc độ tối đa cho T4 GPU
os.environ["HACHIMIMT_BATCH_SIZE"] = "96"             # Batch 96 tối ưu tensor cores T4 (15GB VRAM)
os.environ["HACHIMIMT_CT2_BATCH_TYPE"] = "tokens"       # Ghép theo tokens giảm độ trễ
os.environ["HACHIMIMT_CT2_WINDOW_MULTIPLIER"] = "16"   # Tăng 25-31% tốc độ trên T4 x1
os.environ["HACHIMIMT_INTER_THREADS"] = "1"
os.environ["HACHIMIMT_AUTO_ALL_GPUS"] = "1"            # Tự động nhận diện nếu có 2 GPU (Kaggle/Colab Pro)
os.environ["HACHIMIMT_FORCE_CT2_CUDA"] = "1"           # Ép CT2 chạy CUDA, không fallback CPU
os.environ["HACHIMIMT_COMPUTE_TYPE"] = "int8_float16"  # INT8 tính toán kết hợp FP16 siêu tốc
os.environ["HACHIMIMT_TOKENIZE_WORKERS"] = str(min(os.cpu_count() or 4, 8))
os.environ["ALLOWED_HOSTS"] = "*"                      # Cho phép tunnel Cloudflare / ngrok
os.environ["ALLOW_TUNNEL"] = "1"
os.environ["COLAB_GPU"] = "1"

# 4. Kích hoạt preload CUDA trong session Colab hiện tại
try:
    from translation.core.ct2_safe_import import preload_cuda_libraries
    preload_cuda_libraries()
    print("✅ Đã nạp thành công libcublas.so.12 vào bộ nhớ tiến trình Colab!")
except Exception as e:
    print(f"⚠️ Preload CUDA: {e}")

### 3. Cài đặt thư viện phụ thuộc & Cloudflare Tunnel (~1–2 phút)

In [ ]:
# ==============================================================================
# BƯỚC 3: CÀI ĐẶT CÁC THƯ VIỆN & TẢI CLOUDFLARED TUNNEL
# ==============================================================================
print("📦 Đang cài đặt thư viện từ requirements.txt...")
!pip install -q -r requirements.txt

# Cài đặt cloudflared để tạo link truy cập web HTTPS miễn phí (hỗ trợ WebSocket đầy đủ)
import urllib.request
CLOUDFLARED_PATH = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED_PATH):
    print("🌐 Đang tải Cloudflare Tunnel (cloudflared)...)")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED_PATH
    )
    os.chmod(CLOUDFLARED_PATH, 0o755)

# Cài đặt ffmpeg cho xử lý gộp audio
!apt-get install -y -qq ffmpeg

# Kiểm tra CTranslate2 CUDA
import ctranslate2
cuda_count = ctranslate2.get_cuda_device_count()
print(f"\n🔍 CTranslate2 CUDA Device Count: {cuda_count}")
if cuda_count > 0:
    supported_types = ctranslate2.get_supported_compute_types("cuda")
    print(f"✅ CTranslate2 đã kết nối CUDA thành công! Các kiểu tính toán hỗ trợ: {supported_types}")
else:
    print("⚠️ CTranslate2 chưa nhận thấy GPU CUDA! Đang chạy chế độ CPU.")

### 4. Tải sẵn Model dịch & Giọng đọc Piper TTS (Khuyên dùng)

Tải trước model dịch khuyên dùng (`HachimiMT-60-QT`) và các giọng đọc tiếng Việt hay nhất để không phải chờ đợi khi bấm dịch lần đầu.

In [ ]:
# ==============================================================================
# BƯỚC 4: TẢI SẴN MODEL DỊCH & GIỌNG ĐỌC AUDIO
# ==============================================================================
# 1. Tải model dịch HachimiMT-60-QT (khoảng 58 MB, chỉ mất vài giây)
print("📥 Đang kiểm tra & tải model dịch HachimiMT-60-QT...")
!python scripts/download_model.py --model HachimiMT-60-QT --smoke

# 2. Tải các giọng đọc Piper TTS tiếng Việt
print("\n🎙️ Đang kiểm tra & tải các giọng đọc Piper TTS tiếng Việt...")
!python scripts/download_voices.py

### 5. Khởi chạy Web UI & Mở đường link công khai (*.trycloudflare.com)

Cell này sẽ khởi động Web Server và in ra đường link công khai (`https://*.trycloudflare.com`). Bấm vào link để mở giao diện dịch truyện!

In [ ]:
# ==============================================================================
# BƯỚC 5: KHỞI CHẠY BACKEND & CLOUDFLARE TUNNEL
# ==============================================================================
import os, sys, time, re, subprocess
from IPython.display import display, HTML

PORT = 8000

# 1. Khởi chạy Uvicorn Backend trong tiến trình nền
print("🚀 Đang khởi động Web Backend...")
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 2. Khởi chạy Cloudflare Tunnel để tạo URL công khai
print("🌐 Đang khởi tạo đường truyền Cloudflare Tunnel...")
CLOUDFLARED_BIN = "/content/cloudflared" if os.path.exists("/content/cloudflared") else "cloudflared"
tunnel_process = subprocess.Popen(
    [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Đọc log của tunnel để trích xuất URL public
public_url = None
start_time = time.time()
while time.time() - start_time < 45:
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #1e1e2f, #2d3748); border: 2px solid #4fd1c5; border-radius: 12px; padding: 20px; margin: 15px 0; text-align: center; color: white; font-family: sans-serif;">
        <h2 style="color: #4fd1c5; margin-top: 0;">🎉 NOVELREADER ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 15px; margin-bottom: 20px;">Bấm vào nút bên dưới để mở giao diện Dịch truyện & TTS Audio trên trình duyệt:</p>
        <a href="{public_url}" target="_blank" style="background: #4fd1c5; color: #1a202c; padding: 12px 28px; font-size: 17px; font-weight: bold; border-radius: 8px; text-decoration: none; display: inline-block; box-shadow: 0 4px 14px rgba(79, 209, 197, 0.4);">
            👉 MỞ GIAO DIỆN NOVELREADER (BẤM VÀO ĐÂY) 👈
        </a>
        <p style="font-size: 13px; color: #a0aec0; margin-top: 15px;">Đường dẫn: <code style="color: #63b3ed;">{public_url}</code></p>
    </div>
    """))
    print(f"\n✅ Public Link: {public_url}\n")
else:
    print("⚠️ Không lấy được URL Cloudflare tự động. Đang kiểm tra log tunnel...")

# 4. Stream log của Web Server (nhấn nút Stop trên cell để dừng ứng dụng)
print("📋 Đang theo dõi log tiến trình (Bấm nút ⏹ Stop để tắt ứng dụng):\n")
try:
    while True:
        line = server_process.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.2)
except KeyboardInterrupt:
    print("\n⏹ Đang dừng ứng dụng...")
    server_process.terminate()
    tunnel_process.terminate()
    print("✅ Đã dừng an toàn.")